# core.models

> Data models for virtual collection state, configuration, column definitions, render contexts, and URL bundles.

In [ ]:
#| default_exp core.models

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from dataclasses import dataclass, field
from typing import Optional, Tuple

## Auto-Prefix Generator

In [ ]:
#| export
_prefix_counter: int = 0

def _auto_prefix() -> str:  # Auto-generated prefix like 'vc0', 'vc1', etc.
    """Generate a unique prefix for a virtual collection instance."""
    global _prefix_counter
    p = f"vc{_prefix_counter}"
    _prefix_counter += 1
    return p

## ColumnDef

In [ ]:
#| export
@dataclass
class ColumnDef:
    """Column definition for table layout."""
    key: str                            # Unique column identifier (used in cell IDs)
    header: str = ""                    # Display text for header
    width: str = "1fr"                  # CSS grid column width
    sortable: bool = False              # Whether column header is clickable for sort
    resizable: bool = False             # Whether column border is draggable
    header_cls: str = ""                # Additional CSS classes for header cell
    cell_cls: str = ""                  # Additional CSS classes for data cells

## VirtualCollectionConfig

In [ ]:
#| export
@dataclass
class VirtualCollectionConfig:
    """Initialization-time configuration for a virtual collection."""
    prefix: str = ""                                    # HTML ID prefix (auto-generated if empty)
    layout: str = "table"                               # 'table' or 'grid'
    row_height: int = 40                                # Row height in px (fixed)

    # Table layout
    columns: Tuple[ColumnDef, ...] = ()                 # Column definitions
    sticky_left: int = 0                                # Number of columns pinned left
    sticky_right: int = 0                               # Number of columns pinned right

    # Grid layout
    columns_per_row: int = 4                            # Items per grid row
    grid_gap: str = "1rem"                              # Gap between grid items

    # Scroll/touch
    disable_scroll_in_modes: Tuple[str, ...] = ()       # Mode-based scroll suppression

    # Scrollbar
    show_scrollbar: bool = True                         # Show custom scrollbar
    min_thumb_height: int = 24                          # Minimum scrollbar thumb height (px)

    def __post_init__(self):
        if not self.prefix:
            self.prefix = _auto_prefix()

## VirtualCollectionState

In [ ]:
#| export
@dataclass
class VirtualCollectionState:
    """Mutable runtime state for a virtual collection."""
    window_start: int = 0               # First visible row index
    visible_rows: int = 0               # Number of visible rows (from auto-fit)
    total_items: int = 0                # Total item count (set by consumer)
    viewport_height: float = 0.0        # Measured viewport height (px)
    is_auto_mode: bool = True           # Auto-adjust visible rows from viewport
    cursor_index: int = -1              # Keyboard cursor position (-1 = none)
    sort_column: str = ""               # Current sort column key (empty = unsorted)
    sort_ascending: bool = True         # Sort direction

## Render Contexts

In [ ]:
#| export
@dataclass
class RowRenderContext:
    """Context passed to row/item render callback."""
    index: int                          # Row index in the full collection
    total_items: int                    # Total item count
    is_cursor: bool = False             # Whether this row is the keyboard cursor
    is_first_visible: bool = False      # First row in current window
    is_last_visible: bool = False       # Last row in current window

In [ ]:
#| export
@dataclass
class CellRenderContext:
    """Context passed to cell render callback."""
    row_index: int                      # Row index in the full collection
    column: ColumnDef                   # Column definition
    total_items: int                    # Total item count
    is_cursor: bool = False             # Whether this row is the keyboard cursor

## VirtualCollectionUrls

In [ ]:
#| export
@dataclass
class VirtualCollectionUrls:
    """URL bundle for HTMX endpoints."""
    # Navigation
    nav_up: str = ""                    # Navigate up one row
    nav_down: str = ""                  # Navigate down one row
    nav_page_up: str = ""               # Navigate up one page
    nav_page_down: str = ""             # Navigate down one page
    nav_first: str = ""                 # Navigate to first row
    nav_last: str = ""                  # Navigate to last row
    nav_to_index: str = ""              # Navigate to specific row index

    # Viewport
    update_viewport: str = ""           # Update visible_rows (auto-fit)

    # Focus
    focus_row: str = ""                 # Move cursor to a specific row (click/tap)

    # Actions
    activate: str = ""                  # Activate focused row (Space/Enter)

    # Sort
    sort: str = ""                      # Sort by column (header click)

## Tests

In [ ]:
# Test auto-prefix generation
config1 = VirtualCollectionConfig()
config2 = VirtualCollectionConfig()
assert config1.prefix != config2.prefix
print(f"Auto-prefixes: {config1.prefix}, {config2.prefix}")

Auto-prefixes: vc0, vc1


In [ ]:
# Test explicit prefix
config = VirtualCollectionConfig(prefix="fb")
assert config.prefix == "fb"
print(f"Explicit prefix: {config.prefix}")

Explicit prefix: fb


In [ ]:
# Test ColumnDef
col = ColumnDef(key="name", header="Name", width="1fr", sortable=True)
assert col.key == "name"
assert col.sortable == True
print(f"Column: {col.key}, width={col.width}, sortable={col.sortable}")

Column: name, width=1fr, sortable=True


In [ ]:
# Test state defaults
state = VirtualCollectionState()
assert state.window_start == 0
assert state.cursor_index == -1
assert state.sort_column == ""
assert state.is_auto_mode == True
print(f"State: window_start={state.window_start}, cursor={state.cursor_index}")

State: window_start=0, cursor=-1


In [ ]:
# Test render contexts
col = ColumnDef(key="name", header="Name")
row_ctx = RowRenderContext(index=5, total_items=500, is_cursor=True)
cell_ctx = CellRenderContext(row_index=5, column=col, total_items=500, is_cursor=True)
assert row_ctx.is_cursor == True
assert cell_ctx.column.key == "name"
print(f"Row context: index={row_ctx.index}, cursor={row_ctx.is_cursor}")
print(f"Cell context: row={cell_ctx.row_index}, col={cell_ctx.column.key}")

Row context: index=5, cursor=True
Cell context: row=5, col=name


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()